# Задание 4
# Многомерная регрессия
**Цель работы:** изучение принципов решения задачи многомерной регрессии с использованием методов машинного обучения.

## Транспортные средства от CarDekho

Полезный набор данных для прогнозирования цен. Этот набор данных включает сведения о подержанных автомобилях и мотоциклах на CarDekho.com. Эти данные могут быть использованы для множества целей, таких как прогнозирование цен,чтобы проиллюстрировать использование линейной регрессии в машинном обучении.
Атрибуты:
* **name** – модель
* **year** – год
* **selling_price** – цена продажи
* **km_driven** – пробег в километрах
* **fuel** – тип топлива (Petrol, Diesel, CNG)
* **seller_type** – тип продавца (Dealer, Individual)
* **transmission** – тип трансмиссии (Manual, Automatic)
* **owner** – количество предыдущих владельцев (0, 1, 3)





### Импортируем библиотеки и загружаем данные

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/datasets/car data.csv')
df.head()

### Получим информацию о датасете


In [ ]:
df.info()

### Проверяем наличие пропущенных значений и выбросов


In [ ]:
# проверяем пропущенные значения
print(df.isnull().sum())

### Заменяем категориальные значения числовыми

In [ ]:
df_copy = df.copy()

In [ ]:
# удаляем ненужный признак
df_copy = df_copy.drop('name', axis=1)

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder_fuel = LabelEncoder()
df_copy['fuel'] = labelencoder_fuel.fit_transform(df_copy['fuel'])

labelencoder_seller_type = LabelEncoder()
df_copy['seller_type'] = labelencoder_seller_type.fit_transform(df_copy['seller_type'])

labelencoder_transmission = LabelEncoder()
df_copy['transmission'] = labelencoder_transmission.fit_transform(df_copy['transmission'])

labelencoder_owner = LabelEncoder()
df_copy['owner'] = labelencoder_owner.fit_transform(df_copy['owner'])
df_copy

### Разделяем данные на признаки и целевую переменную

In [ ]:
y = df_copy['selling_price']
X = df_copy.drop('selling_price', axis=1)

In [ ]:
# убедимся, что данные в нужном нам формате
type(X), type(y)

In [ ]:
# посмотрим на признаки
X.head()

### Разделяем данные на обучающую и тестовую выборку

75% данных используется для обучения и 25% – для тестирования.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

### Применяем операцию нормализации для численной устойчивости

In [ ]:
# импортируем класс для стандартизации данных
from sklearn.preprocessing import StandardScaler
# и создания модели линейной регрессии
from sklearn.linear_model import LinearRegression

# создадим объект класса StandardScaler
scaler = StandardScaler()
scaler

### Масштабируем признаки обучающей выборки


In [ ]:
X_train_scaled = scaler.fit_transform(X_train)

# убедимся, что объект scaler запомнил значения среднего и СКО для каждого признака
scaler.mean_, scaler.scale_

### Обучаем модель линейной регрессии

In [ ]:
# применим масштабированные данные для обучения модели линейной регрессии
model = LinearRegression().fit(X_train_scaled, y_train)
model

### Делаем прогноз на основе данных тестирования

In [ ]:
# преобразуем тестовые данные с использованием среднего и СКО, рассчитанных на обучающей выборке
# так тестовые данные не повлияют на обучение модели, и мы избежим утечки данных
X_test_scaled = scaler.transform(X_test)

# сделаем прогноз на стандартизированных тестовых данных
y_pred = model.predict(X_test_scaled)
# выведем первые пять значений с помощью диапазона индексов
y_pred[:5]

In [ ]:
# импортируем функцию корня среднеквадратической ошибки
from sklearn.metrics import root_mean_squared_error

# сравним тестовые и прогнозные значения
print('Root Mean Squared Error (RMSE):', root_mean_squared_error(y_test, y_pred))

In [ ]:
# посмотрим на еще одну метрику - коэффициент детерминации R2
from sklearn.metrics import r2_score
print('R2:', np.round(r2_score(y_test, y_pred), 2))

In [ ]:
# оценим R-квадрат (метрика (score) по умолчанию для класса LinearRegression)
model.score(X_test_scaled, y_test)

### Итоговое уравнение

In [ ]:
# коэффициенты
print('Coefficients:', model.coef_)

In [ ]:
# свободный член
print('Intercept:', model.intercept_)

### Оценим влияние признаков на целевую переменную

In [ ]:
from sklearn.preprocessing import LabelEncoder
df = df.drop('name', axis=1)

#df = df.drop(['name', 'seller_type', 'km_driven'], axis=1)

labelencoder_fuel2 = LabelEncoder()
df['fuel'] = labelencoder_fuel2.fit_transform(df['fuel'])

labelencoder_seller_type2 = LabelEncoder()
df['seller_type'] = labelencoder_seller_type2.fit_transform(df['seller_type'])

labelencoder_transmission2 = LabelEncoder()
df['transmission'] = labelencoder_transmission2.fit_transform(df['transmission'])

labelencoder_owner2 = LabelEncoder()
df['owner'] = labelencoder_owner2.fit_transform(df['owner'])
df

In [ ]:
df.corr()

### Визуализация результатов регрессии

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Фактические значения')
plt.ylabel('Предсказанные значения')
plt.title('Фактические vs Предсказанные значения')
plt.grid()
plt.show()

### Остатки регрессии (Residuals Plot)

Остаток — это разница между фактическим и предсказанным значением (y_true - y_pred).
Этот график помогает проверить важное предположение линейной регресии: что остатки случайны и не имеют паттернов.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Нет ошибки')
plt.xlabel('Предсказанные значения (Predicted)')
plt.ylabel('Остатки (Residuals)')
plt.title('Диаграмма остатков (Residuals Plot)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Результат:
* Остатки беспорядочно разбросаны вокруг горизонтальной красной линии (нуля), нет никаких явных дуг, форм или конусов.